# Anthropic Claude via APIM AI Gateway

This notebook tests Claude models hosted on Azure AI Foundry and proxied through Azure API Management.

## Setup

1. Deploy `options-infra/ai-gateway-anthropic` with `azd up`
2. In this directory, run `uv sync`
3. In VSCode: `Ctrl+Shift+P` → **Python: Select Interpreter** → pick `./agents_anthropic/.venv/bin/python`
4. Copy `.env.example` to `.env` and fill in your values

In [28]:
import os
import json
import requests
import anthropic
from dotenv import load_dotenv

load_dotenv(override=True)

APIM_GATEWAY_URL    = os.environ["APIM_GATEWAY_URL"].rstrip("/")
APIM_SUBSCRIPTION_KEY = os.environ["APIM_SUBSCRIPTION_KEY"]
ANTHROPIC_MODEL     = os.environ.get("ANTHROPIC_MODEL", "claude-opus-4-6")

ANTHROPIC_BASE_URL  = f"{APIM_GATEWAY_URL}/inference/anthropic"
MESSAGES_URL        = f"{ANTHROPIC_BASE_URL}/v1/messages"
APIM_HEADERS        = {"api-key": APIM_SUBSCRIPTION_KEY}

print(f"Gateway : {APIM_GATEWAY_URL}")
print(f"Endpoint: {MESSAGES_URL}")
print(f"Model   : {ANTHROPIC_MODEL}")
print(f"Api Key  : {'*' * len(APIM_SUBSCRIPTION_KEY)}")

Gateway : https://apim-rlctgbiul4kly.azure-api.net
Endpoint: https://apim-rlctgbiul4kly.azure-api.net/inference/anthropic/v1/messages
Model   : claude-opus-4-6
Api Key  : ********************************


## 1 · Raw HTTP request

Lowest-level test — confirms APIM is reachable and the API key flow works end-to-end.

In [29]:
payload = {
    "model": ANTHROPIC_MODEL,
    "max_tokens": 256,
    "temperature": 0.7,
    "messages": [{"role": "user", "content": "Say hello and tell me what model you are."}],
    "system": "You are a helpful assistant that provides concise answers to the user's questions."
}

resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01"},
    json=payload,
    timeout=30,
)

print(f"Status : {resp.status_code}")
print(f"Headers: x-request-id={resp.headers.get('x-request-id', 'n/a')}")

body = resp.json()
print(json.dumps(body, indent=2))
print(resp.headers)

Status : 200
Headers: x-request-id=n/a
{
  "model": "claude-opus-4-6",
  "id": "msg_014rhaTMMSaaTk6v6fQTfqEC",
  "type": "message",
  "role": "assistant",
  "content": [
    {
      "type": "text",
      "text": "Hello! I'm Claude, an AI assistant made by Anthropic. Nice to meet you! How can I help you today?"
    }
  ],
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "stop_details": null,
  "usage": {
    "input_tokens": 34,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "cache_creation": {
      "ephemeral_5m_input_tokens": 0,
      "ephemeral_1h_input_tokens": 0
    },
    "output_tokens": 30,
    "service_tier": "standard",
    "inference_geo": "not_available"
  }
}
{'Content-Type': 'application/json', 'Date': 'Tue, 26 May 2026 04:49:11 GMT', 'Transfer-Encoding': 'chunked', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'request-context': 'appId=', 'Request-Id': 'req_011CbQaPtaMWZZzUqHpjgxHd', 'x-robots-tag': 'none'

## 2 · Anthropic Python SDK

Uses the official `anthropic` package pointed at the APIM gateway.
The APIM subscription key replaces the normal Anthropic API key.

In [25]:
client = anthropic.Anthropic(
    base_url=ANTHROPIC_BASE_URL,
    api_key=APIM_SUBSCRIPTION_KEY,   # APIM subscription key, NOT your Anthropic API key
    default_headers=APIM_HEADERS,
)

message = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=512,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."},
    ],
)

print(f"Stop reason  : {message.stop_reason}")
print(f"Input tokens : {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print()
print(message.content[0].text)

AuthenticationError: Error code: 401 - {'statusCode': 401, 'message': 'Access denied due to missing subscription key. Make sure to include subscription key when making requests to an API.'}

## 3 · System prompt + multi-turn conversation

In [ ]:
conversation = [
    {"role": "user", "content": "My name is Alex. Remember it."},
]

turn1 = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=256,
    system="You are a helpful assistant with a great memory.",
    messages=conversation,
)
print("Turn 1:", turn1.content[0].text)

# Add assistant reply and follow-up
conversation.append({"role": "assistant", "content": turn1.content[0].text})
conversation.append({"role": "user", "content": "What's my name?"})

turn2 = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=256,
    system="You are a helpful assistant with a great memory.",
    messages=conversation,
)
print("Turn 2:", turn2.content[0].text)

## 4 · Streaming response

In [ ]:
print("Streaming response:\n")

with client.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "Count from 1 to 10, one number per line."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n[stream complete]")

### 4b · Raw HTTP SSE streaming

Parsing the raw server-sent-event stream from APIM — shows the exact Anthropic SSE wire format.

In [ ]:
import sseclient  # pip: sseclient-py

sse_resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
    json={
        "model": ANTHROPIC_MODEL,
        "max_tokens": 128,
        "stream": True,
        "messages": [{"role": "user", "content": "Count to 5, one number per line."}],
    },
    stream=True,
)

print(f"Status: {sse_resp.status_code}")
print(f"Content-Type: {sse_resp.headers.get('content-type')}\n")

full_text = []
for event in sseclient.SSEClient(sse_resp).events():
    if event.data == "[DONE]" or not event.data:
        continue
    data = json.loads(event.data)
    etype = data.get("type", "")
    if etype == "content_block_delta":
        chunk = data["delta"].get("text", "")
        full_text.append(chunk)
        print(chunk, end="", flush=True)
    elif etype == "message_start":
        print(f"[message_start] model={data['message']['model']}")
    elif etype in ("message_delta", "message_stop"):
        print(f"\n[{etype}]", data.get("delta") or "")

print("\n\nFull text:", "".join(full_text))


## 5 · Vision (image input)

Claude supports image inputs via base64. Here we send a small test PNG.

In [ ]:
import base64, io

# Tiny 1×1 red PNG for smoke-testing vision input
red_pixel_b64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwADhQGAWjR9awAAAABJRU5ErkJggg=="
)

message = client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=128,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": red_pixel_b64,
                    },
                },
                {"type": "text", "text": "What colour is the pixel in this image?"},
            ],
        }
    ],
)
print(message.content[0].text)

## 6 · APIM diagnostics

Check response headers to confirm the request was handled by APIM and inspect rate-limit state.

In [ ]:
resp = requests.post(
    MESSAGES_URL,
    headers={**APIM_HEADERS, "anthropic-version": "2023-06-01"},
    json={
        "model": ANTHROPIC_MODEL,
        "max_tokens": 64,
        "messages": [{"role": "user", "content": "ping"}],
    },
    timeout=30,
)

interesting_headers = [
    "x-request-id",
    "x-ms-region",
    "x-ratelimit-limit-requests",
    "x-ratelimit-remaining-requests",
    "x-ratelimit-limit-tokens",
    "x-ratelimit-remaining-tokens",
    "apim-request-id",
]

print(f"Status: {resp.status_code}\n")
print("Response headers:")
for h in interesting_headers:
    val = resp.headers.get(h)
    if val:
        print(f"  {h}: {val}")

print("\nAll response headers:")
for k, v in resp.headers.items():
    print(f"  {k}: {v}")

## 7 · Foundry Agent with Claude

Requires `AZURE_AI_FOUNDRY_CONNECTION_STRING` in `.env`.  
The Anthropic APIM connection is registered automatically by `azd up` — the agent uses it to route Claude calls through APIM.

Cells in this section:
- **7a** — Connect to project & inspect connections
- **7b** — Create a Claude agent
- **7c** — Single-turn question
- **7d** — Multi-turn conversation
- **7e** — Agent with tools (code interpreter)
- **7f** — Cleanup

### 7a · Connect and find the Anthropic APIM connection

In [ ]:
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential

connection_string = os.environ.get("AZURE_AI_FOUNDRY_CONNECTION_STRING", "")
if not connection_string:
    raise EnvironmentError("AZURE_AI_FOUNDRY_CONNECTION_STRING not set in .env")

aio_creds = AsyncDefaultAzureCredential()
project = AsyncAIProjectClient(endpoint=connection_string, credential=aio_creds)

anthropic_connection = None
print("Connections:")
async for conn in project.connections.list():
    print(f"  {conn.name:55s} type={conn.connection_type}")
    if conn.connection_type == "ApiManagement" and "anthropic" in conn.name.lower():
        anthropic_connection = conn.name
        print(f"  └─ ✅ Using this for agents")

if not anthropic_connection:
    raise RuntimeError("No Anthropic ApiManagement connection found in this project")

AGENT_MODEL = f"{anthropic_connection}/{ANTHROPIC_MODEL}"
print(f"\nAgent model string: {AGENT_MODEL}")


### 7b · Create a v2 agent

In [ ]:
AGENT_NAME = "claude-anthropic-gateway-agent"

# Delete existing agent with the same name to avoid version accumulation
async for existing in project.agents.list():
    if existing.name == AGENT_NAME:
        await project.agents.delete(agent_name=AGENT_NAME)
        print(f"Deleted existing agent '{AGENT_NAME}'")
        break

agent = await project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=AGENT_MODEL,
        instructions="You are a helpful assistant powered by Claude via Azure AI Foundry and APIM. Be concise.",
    ),
)
print(f"Created agent  name={agent.name}  version={agent.version}  model={agent.definition.model}")


### 7c · Single-turn question

In [ ]:
openai_client = project.get_openai_client()

conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "What are the three primary colours? One sentence."}],
)

response = await openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="",
)
print("Agent:", response.output_text)
assert response.output_text, "Expected non-empty reply"
print("✅ Single-turn agent call succeeded")


### 7d · Multi-turn conversation on a single thread

In [ ]:
turns = [
    "My name is Alex. Remember it.",
    "What is the capital of France?",
    "What is my name?",
]

conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": turns[0]}],
)

for i, user_msg in enumerate(turns):
    response = await openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input=user_msg if i > 0 else "",
    )
    print(f"User : {user_msg}")
    print(f"Agent: {response.output_text}")
    print()


### 7e · Streaming response

In [ ]:
conversation = await openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "Tell me hi in 5 random languages."}],
)

stream = await openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="",
    stream=True,
)

print("Streaming: ", end="")
async for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print(f"\n✅ Done  usage={event.response.to_dict()['usage']}")


### 7f · Cleanup

In [ ]:
await project.agents.delete(agent_name=AGENT_NAME)
print(f"Deleted agent '{AGENT_NAME}'")
await project.close()
await aio_creds.close()


## 8 · OpenAI SDK via OpenAI-compatible Anthropic endpoint

APIM also exposes an **OpenAI Chat Completions-compatible** API at `inference/openai-compat`.
This lets you swap in Claude without changing any OpenAI SDK code.

Install the OpenAI SDK: `uv add openai`

In [ ]:
from openai import OpenAI

# Point the OpenAI SDK at APIM; use the APIM subscription key as api_key.
# The compat layer translates Chat Completions → Anthropic Messages transparently.
compat_client = OpenAI(
    base_url=f"{APIM_GATEWAY_URL}/inference/openai-compat",
    api_key=APIM_SUBSCRIPTION_KEY,
    default_headers=APIM_HEADERS,
)

response = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is 2 + 2? Answer in one word."},
    ],
    max_tokens=32,
)

print("Model   :", response.model)
print("Finish  :", response.choices[0].finish_reason)
print("Tokens  :", response.usage)
print()
print(response.choices[0].message.content)


In [ ]:
# Multi-turn conversation — system prompt extraction is handled by the APIM policy
messages = [
    {"role": "system",    "content": "You are a pirate who answers every question as a pirate would."},
    {"role": "user",      "content": "Where is the nearest treasure?"},
]

r1 = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL, messages=messages, max_tokens=128
)
print("Pirate:", r1.choices[0].message.content)

messages.append({"role": "assistant", "content": r1.choices[0].message.content})
messages.append({"role": "user",      "content": "How do I get there?"})

r2 = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL, messages=messages, max_tokens=128
)
print("Pirate:", r2.choices[0].message.content)


### 8a · Setup — OpenAI-compat endpoint URL

In [ ]:
COMPAT_BASE_URL = f"{APIM_GATEWAY_URL}/inference/openai-compat"
COMPAT_CHAT_URL = f"{COMPAT_BASE_URL}/chat/completions"
print("Compat endpoint:", COMPAT_CHAT_URL)

compat_client = OpenAI(
    base_url=COMPAT_BASE_URL,
    api_key=APIM_SUBSCRIPTION_KEY,
    default_headers=APIM_HEADERS,
)


### 8b · Raw HTTP — inspect translated request/response

In [ ]:
# Call the compat endpoint with raw requests so we can inspect the full response shape.
raw_resp = requests.post(
    COMPAT_CHAT_URL,
    headers={**APIM_HEADERS, "Content-Type": "application/json"},
    json={
        "model": ANTHROPIC_MODEL,
        "messages": [{"role": "user", "content": "Say 'hello' and nothing else."}],
        "max_tokens": 32,
    },
)
print("Status :", raw_resp.status_code)
print("Headers:", dict(raw_resp.headers))
print()
body = raw_resp.json()
print(json.dumps(body, indent=2))

# Verify response is in OpenAI format
assert body.get("object") == "chat.completion", f"Expected 'chat.completion', got {body.get('object')}"
assert "choices" in body, "Missing 'choices' field"
assert "usage" in body, "Missing 'usage' field"
assert "prompt_tokens" in body["usage"], "Missing prompt_tokens in usage"
print("\n✅ Response is valid OpenAI Chat Completions format")


### 8c · System prompt extraction

The APIM policy extracts the `system` role message and moves it to Anthropic's top-level `system` field.

In [ ]:
resp = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[
        {"role": "system",    "content": "You only respond in French, no matter what."},
        {"role": "user",      "content": "What is the capital of France?"},
    ],
    max_tokens=64,
)
print(resp.choices[0].message.content)
print("\nFinish reason:", resp.choices[0].finish_reason)


### 8d · Stop sequences mapping

OpenAI `stop` (string or array) → Anthropic `stop_sequences` (array only). The policy handles both.

In [ ]:
# Single stop string — policy wraps in array for Anthropic
r1 = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 10, one number per line."}],
    max_tokens=128,
    stop="5",  # Should stop before or at '5'
)
print("Stop=string result:")
print(r1.choices[0].message.content)
print("Finish reason:", r1.choices[0].finish_reason)
print()

# Array stop
r2 = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[{"role": "user", "content": "List the days of the week."}],
    max_tokens=64,
    stop=["Wednesday", "Thursday"],
)
print("Stop=array result:")
print(r2.choices[0].message.content)
print("Finish reason:", r2.choices[0].finish_reason)


### 8e · Usage / token mapping

Verifies that Anthropic's `input_tokens`/`output_tokens` are correctly mapped to OpenAI's `prompt_tokens`/`completion_tokens`.

In [ ]:
resp = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=16,
)
u = resp.usage
print(f"prompt_tokens     : {u.prompt_tokens}")
print(f"completion_tokens : {u.completion_tokens}")
print(f"total_tokens      : {u.total_tokens}")
assert u.total_tokens == u.prompt_tokens + u.completion_tokens
assert u.prompt_tokens > 0 and u.completion_tokens > 0
print("\n✅ Token counts correctly mapped from Anthropic → OpenAI format")


### 8f · Streaming via OpenAI-compat endpoint

The compat policy forces `stream=false` to Anthropic (real SSE passthrough isn't possible because the two SSE schemas differ), then re-encodes the full response as a valid OpenAI SSE stream before returning to the client — **pseudo-streaming**: content arrives in one chunk rather than token-by-token, but the OpenAI SDK streaming API works correctly.

In [ ]:
# Streaming via compat endpoint — content arrives as one SSE chunk.
stream = compat_client.chat.completions.create(
    model=ANTHROPIC_MODEL,
    messages=[
        {"role": "system", "content": "You are concise."},
        {"role": "user",   "content": "List 3 planets, one per line."},
    ],
    max_tokens=128,
    stream=True,
)

print("Streaming chunks:")
full_content = []
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        full_content.append(delta)
        print(repr(delta))

print("\nFull response:", "".join(full_content))
assert "".join(full_content), "Expected non-empty streaming content"
print("✅ Streaming works on compat endpoint (pseudo-streaming: full content in one chunk)")
